# RQ analysis
**Q1**: Spearman rank correlation of latency/energy across the 3 input domains, per framework.

**Q2**: Spearman rank correlation of latency/energy across the 3 frameworks, per input domain.

**Q3**: Jaccard similarity of Pareto-optimal arch sets across (task, framework); shared subset.

Assumes `results/latency_rpi.csv` has columns `lat_ms_median` and (when measured) `energy_mj_median`. Accuracy pulled from HW-NAS-Bench pickle (cifar100 only for now).

In [ ]:
from pathlib import Path
import sys, pickle, warnings, itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
CSV = ROOT / 'results' / 'latency_rpi.csv'
HW_PICKLE = ROOT / 'data' / 'hw-nas-bench' / 'HW-NAS-Bench-v1_0.pickle'

df = pd.read_csv(CSV)
df = df[df.status == 'ok'].copy()
TASKS = sorted(df.task.unique())
RUNTIMES = sorted(df.runtime.unique())
HAS_ENERGY = 'energy_mj_median' in df.columns
print(f'rows: {len(df)}  archs: {df.arch_idx.nunique()}  tasks: {TASKS}  runtimes: {RUNTIMES}  energy_col: {HAS_ENERGY}')

## Helpers

In [ ]:
def pivot_metric(df, metric, index='arch_idx', col_a='task', col_b='runtime'):
    """Return wide df: rows = arch, cols = (col_a, col_b), vals = metric."""
    return df.pivot_table(index=index, columns=[col_a, col_b], values=metric)

def spearman_matrix(wide):
    """Spearman correlation matrix over columns of `wide`. NaN-safe via pairwise."""
    cols = list(wide.columns)
    n = len(cols)
    M = np.full((n, n), np.nan)
    for i, j in itertools.product(range(n), range(n)):
        a, b = wide[cols[i]], wide[cols[j]]
        mask = a.notna() & b.notna()
        if mask.sum() < 3: continue
        rho, _ = spearmanr(a[mask], b[mask])
        M[i, j] = rho
    return pd.DataFrame(M, index=cols, columns=cols)

def heatmap(M, title, ax=None, vmin=-1, vmax=1):
    if ax is None: fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(M.values, vmin=vmin, vmax=vmax, cmap='RdBu_r')
    ax.set_xticks(range(len(M.columns))); ax.set_xticklabels(M.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(M.index))); ax.set_yticklabels(M.index)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                        color='white' if abs(v) > 0.5 else 'black')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

## Q1: rank correlation across tasks, per framework
For each framework, take latency (and energy) per arch across the 3 tasks, then Spearman across task pairs.

In [ ]:
def q1(metric, label):
    fig, axes = plt.subplots(1, len(RUNTIMES), figsize=(4.2*len(RUNTIMES), 3.5))
    if len(RUNTIMES) == 1: axes = [axes]
    results = {}
    for ax, rt in zip(axes, RUNTIMES):
        sub = df[df.runtime == rt]
        wide = sub.pivot_table(index='arch_idx', columns='task', values=metric)
        M = spearman_matrix(wide)
        results[rt] = M
        heatmap(M, f'{label} | {rt} (n={len(wide)})', ax=ax)
    plt.tight_layout(); plt.show()
    return results

q1_lat = q1('lat_ms_median', 'latency rank')

In [ ]:
if HAS_ENERGY:
    q1_eng = q1('energy_mj_median', 'energy rank')
else:
    print('energy column missing - skip')

## Q2: rank correlation across frameworks, per task

In [ ]:
def q2(metric, label):
    fig, axes = plt.subplots(1, len(TASKS), figsize=(4.2*len(TASKS), 3.5))
    if len(TASKS) == 1: axes = [axes]
    results = {}
    for ax, t in zip(axes, TASKS):
        sub = df[df.task == t]
        wide = sub.pivot_table(index='arch_idx', columns='runtime', values=metric)
        M = spearman_matrix(wide)
        results[t] = M
        heatmap(M, f'{label} | {t} (n={len(wide)})', ax=ax)
    plt.tight_layout(); plt.show()
    return results

q2_lat = q2('lat_ms_median', 'latency rank')

In [ ]:
if HAS_ENERGY:
    q2_eng = q2('energy_mj_median', 'energy rank')
else:
    print('energy column missing - skip')

## Q3: Pareto-optimal sets across (task, framework)
Pareto: minimize latency (or energy), maximize accuracy. Accuracy from HW-NAS-Bench (cifar100 only; other tasks stubbed).

In [ ]:
if str(HW_PICKLE.parent.parent) not in sys.path: sys.path.insert(0, str(HW_PICKLE.parent.parent))
_VDW = getattr(np, 'VisibleDeprecationWarning', None) or np.exceptions.VisibleDeprecationWarning

NB360_PICKLES = {
    'ninapro': ROOT / 'data' / 'nb360_ninapro' / 'NATS-tss-v1_0-daa55.pickle',
    'darcy':   ROOT / 'data' / 'nb360_darcyflow' / 'NATS-tss-v1_0-48858.pickle',
}
NB360_INTERNAL = {'ninapro': 'ninapro', 'darcy': 'darcyflow'}

def _load_nb360_acc(path, internal_name, epoch=199, seed=777):
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=_VDW)
        with open(path, 'rb') as f: d = pickle.load(f)
    acc = {}
    key = (internal_name, seed)
    target = f'ori-test@{epoch}'
    for idx, info in d['arch2infos'].items():
        try:
            ea = info['200']['all_results'][key]['eval_acc1es']
            acc[int(idx)] = float(ea[target])
        except KeyError:
            continue
    return acc

def accuracy_for(task):
    """Return dict arch_idx -> final-epoch test accuracy."""
    if task in NB360_PICKLES:
        return _load_nb360_acc(NB360_PICKLES[task], NB360_INTERNAL[task])
    if task == 'cifar100':
        return None  # TODO: NB201 .pth
    return None

ACC_BY_TASK = {t: accuracy_for(t) for t in TASKS}
print({t: (None if v is None else f'n={len(v)} range=[{min(v.values()):.3f},{max(v.values()):.3f}]')
       for t,v in ACC_BY_TASK.items()})

In [ ]:
def pareto_front(points):
    """points: array (n, d) to minimize on all dims. Returns boolean mask of non-dominated."""
    P = np.asarray(points)
    n = P.shape[0]
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]: continue
        dom = np.all(P <= P[i], axis=1) & np.any(P < P[i], axis=1)
        if dom.any(): keep[i] = False
    return keep

def pareto_set(task, runtime, metric='lat_ms_median'):
    """Return set of arch_idx on Pareto front for (task, runtime) using (metric, -accuracy).
    Falls back to single-objective (just minimize metric => single point) if no accuracy.
    """
    sub = df[(df.task == task) & (df.runtime == runtime)][['arch_idx', metric]].dropna()
    acc = ACC_BY_TASK.get(task)
    if acc is None:
        # no accuracy -> Pareto on single objective = just the min
        if sub.empty: return set()
        return {int(sub.loc[sub[metric].idxmin(), 'arch_idx'])}
    sub['acc'] = sub.arch_idx.map(acc)
    sub = sub.dropna()
    if sub.empty: return set()
    pts = sub[[metric]].values
    pts = np.hstack([pts, -sub[['acc']].values])  # minimize both
    mask = pareto_front(pts)
    return set(sub.arch_idx.values[mask].astype(int))

In [ ]:
def jaccard(a, b):
    if not a and not b: return np.nan
    return len(a & b) / len(a | b)

def q3(metric, label):
    keys = [(t, rt) for t in TASKS for rt in RUNTIMES]
    sets = {k: pareto_set(*k, metric=metric) for k in keys}
    labels = [f'{t}/{rt}' for t,rt in keys]
    n = len(keys)
    J = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            J[i,j] = jaccard(sets[keys[i]], sets[keys[j]])
    JM = pd.DataFrame(J, index=labels, columns=labels)
    fig, ax = plt.subplots(figsize=(0.6*n+2, 0.6*n+1.5))
    heatmap(JM, f'Jaccard | Pareto({label}, acc)', ax=ax, vmin=0, vmax=1)
    plt.tight_layout(); plt.show()
    common = set.intersection(*sets.values()) if all(sets.values()) else set()
    print(f'sizes: {{k: len(v) for k,v in sets.items()}} =')
    for k,v in sets.items(): print(f'  {k}: |P|={len(v)}')
    print(f'common across all (task, framework): {sorted(common)}')
    return sets, JM, common

q3_lat = q3('lat_ms_median', 'latency')

In [ ]:
if HAS_ENERGY:
    q3_eng = q3('energy_mj_median', 'energy')
else:
    print('energy column missing - skip')